<a href="https://colab.research.google.com/github/Rublevskaya-Nastya/HSE_Rublevskaya_Anastasya/blob/lessen-1/final_hw_efrsb_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ Итоговое домашнее задание
# «Аналитик реестра — Обработка данных ЕФРСБ»





Вы — юрист, анализирующий данные из **Единого федерального реестра сведений о банкротстве (ЕФРСБ)**.
Вам поступила выгрузка в формате JSON с информацией о сообщениях по делам о банкротстве за 2025 год.

**Ваша задача** — извлечь ключевые данные, очистить от мусора, сопоставить с реестром организаций
и сформировать аналитический отчёт для руководства.

---
### 📋 Как выполнять и сдавать задание

**Где выполнять:**
- **Вариант 1 (рекомендуется):** Загрузите файл `.ipynb` в [Google Colab](https://colab.research.google.com/), загрузите папку `final_hw_data/` в среду выполнения (например, через боковую панель «Файлы» или смонтировав Google Drive).
- **Вариант 2:** Работайте локально в Jupyter Notebook / JupyterLab. Убедитесь, что папка `final_hw_data/` находится в той же директории, что и файл задания.

**Как сдавать на проверку:**
- Отправьте **ссылку на Google Colab** с общим доступом («Любой, у кого есть ссылка — комментатор»), **ИЛИ**
- Отправьте **ссылку на GitHub-репозиторий**, в котором лежит выполненный `.ipynb`-файл и папка `final_hw_data/` с результатами.

> ⚠️ Убедитесь, что ссылка открывается в режиме просмотра и все ячейки выполнены (результаты видны).

---

### 📦 Исходные данные

В папке `final_hw_data/` находятся 3 файла:

| Файл | Описание |
|------|----------|
| `bankruptcy_messages.json` | Сообщения ЕФРСБ |
| `organizations.json` | Реестр организаций |
| `priority_cases.txt` | Номера приоритетных дел |



---
# 🟢 Загрузка и кросс-референс (0-2 балла)
---

### Задание 1.1 — Загрузка данных из файлов

Загрузите данные из трёх файлов:
- `bankruptcy_messages.json` → список `messages`
- `organizations.json` → список `organizations`
- `priority_cases.txt` → список `priority_case_numbers`

Выведите количество загруженных записей из каждого источника.

In [ ]:
import json
import os

# Словарь для хранения количества записей
counts = {}

# 1. Загрузка сообщений о банкротстве (bankruptcy_messages.json)
messages_file = 'final_hw_data/bankruptcy_messages.json'
if os.path.exists(messages_file):
    try:
        with open(messages_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        counts['messages'] = len(data) if isinstance(data, list) else 0
    except (json.JSONDecodeError, Exception):
        counts['messages'] = 0
else:
    counts['messages'] = 0

# 2. Загрузка реестра организаций (organizations.json)
orgs_file = 'final_hw_data/organizations.json'
if os.path.exists(orgs_file):
    try:
        with open(orgs_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        counts['organizations'] = len(data) if isinstance(data, list) else 0
    except (json.JSONDecodeError, Exception):
        counts['organizations'] = 0
else:
    counts['organizations'] = 0

# 3. Загрузка приоритетных дел (priority_cases.txt)
priority_file = 'final_hw_data/priority_cases.txt'
counts['priority_cases'] = 0
if os.path.exists(priority_file):
    try:
        with open(priority_file, 'r', encoding='utf-8') as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]
        counts['priority_cases'] = len(lines)
    except Exception:
        counts['priority_cases'] = 0

# Вывод результатов
print(f"Количество сообщений (bankruptcy_messages.json): {counts.get('messages', 0)}")
print(f"Количество организаций (organizations.json): {counts.get('organizations', 0)}")
print(f"Количество приоритетных дел (priority_cases.txt): {counts.get('priority_cases', 0)}")

### Задание 1.2 — Словарь организаций

Создайте словарь `org_by_inn`, где ключ — ИНН (строка), а значение — словарь с данными об организации.
Используйте **dict comprehension**.

Выведите количество организаций в словаре и информацию по ИНН `"7701234567"`.

In [ ]:
import json
import os

# Загрузка данных из файла organizations.json
orgs_file = 'final_hw_data/organizations.json'
organizations = []

if os.path.exists(orgs_file):
    try:
        with open(orgs_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        # Проверяем, что данные — это список
        if isinstance(data, list):
            organizations = data
    except (json.JSONDecodeError, Exception) as e:
        print(f"Ошибка при чтении файла organizations.json: {e}")

# Создание словаря org_by_inn с помощью dict comprehension
# Ключ — строка с ИНН, значение — весь словарь организации
org_by_inn = {str(org.get('inn')): org for org in organizations if org.get('inn') is not None}

# Вывод результатов

# 1. Количество организаций в словаре
print(f"Количество организаций в словаре: {len(org_by_inn)}")

# 2. Информация по ИНН "7701234567"
target_inn = "7701234567"
org_info = org_by_inn.get(target_inn)

if org_info:
    print(f"\nИнформация по ИНН {target_inn}:")
    # Выводим все поля организации в удобном для чтения виде
    for key, value in org_info.items():
        print(f"{key}: {value}")
else:
    print(f"\nОрганизация с ИНН {target_inn} не найдена.")

### Задание 1.3 — Кросс-референс: связываем сообщения с организациями

Для каждого сообщения добавьте поле `org_name` (название организации) и `region`,
используя словарь `org_by_inn` и метод `.get()`.
Если организация не найдена — ставьте значение `None`.

Посчитайте, сколько сообщений удалось связать с организацией,
а сколько — не удалось.

In [ ]:
import json
import os

# --- Загрузка данных ---
messages_file = 'final_hw_data/bankruptcy_messages.json'
messages = []

if os.path.exists(messages_file):
    try:
        with open(messages_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, list):
            # Создаем копию, чтобы не изменять исходный файл на диске
            messages = [msg.copy() for msg in data]
    except (json.JSONDecodeError, Exception) as e:
        print(f"Ошибка при чтении файла bankruptcy_messages.json: {e}")

# --- Подсчет и связывание ---
linked_count = 0
not_linked_count = 0

for msg in messages:
    # Получаем ИНН публикатора из сообщения
    publisher_inn = msg.get('publisher_inn')

    # Ищем организацию в словаре. Если не найдена, get() вернет None
    org_data = org_by_inn.get(str(publisher_inn)) if publisher_inn is not None else None

    if org_data:
        # Если организация найдена, добавляем поля и увеличиваем счетчик
        msg['org_name'] = org_data.get('name')
        msg['region'] = org_data.get('region')
        linked_count += 1
    else:
        # Если не найдена, ставим None и увеличиваем другой счетчик
        msg['org_name'] = None
        msg['region'] = None
        not_linked_count += 1

# --- Вывод результатов ---
print(f"Сообщений успешно связано с организацией: {linked_count}")
print(f"Сообщений не удалось связать: {not_linked_count}")

### Задание 1.4 — Множества и приоритетные дела

1. Создайте множество `priority_set` из списка `priority_case_numbers`.
2. Создайте множество `message_case_set` из номеров дел в `messages`
   (используйте list comprehension + filter для непустых номеров).
3. Найдите пересечение — номера дел, которые есть и в приоритетных, и в сообщениях (`&`).
4. Выведите результат.

In [ ]:
import json
import os

# --- 1. Загрузка списка приоритетных дел ---
priority_case_numbers = []
priority_file = 'final_hw_data/priority_cases.txt'

if os.path.exists(priority_file):
    try:
        with open(priority_file, 'r', encoding='utf-8') as f:
            # Убираем пробелы и пустые строки
            priority_case_numbers = [line.strip() for line in f if line.strip()]
    except Exception as e:
        print(f"Ошибка при чтении priority_cases.txt: {e}")

# --- 2. Создание множества из приоритетных дел ---
priority_set = set(priority_case_numbers)

# --- 3. Создание множества из номеров дел в сообщениях ---
# Используем list comprehension для извлечения и filter для удаления пустых значений
# Затем преобразуем результат в множество
message_case_set = set(
    [msg.get('case_number') for msg in messages
     if msg.get('case_number') and str(msg.get('case_number')).strip() != ""]
)

# --- 4. Поиск пересечения (общие номера дел) ---
intersection_set = priority_set & message_case_set

# --- 5. Вывод результатов ---
print(f"Приоритетных дел загружено: {len(priority_set)}")
print(f"Номеров дел в сообщениях: {len(message_case_set)}")
print(f"\nПересечение (номера дел, которые есть и там, и там):")

if intersection_set:
    # Сортируем для удобства чтения
    for case_number in sorted(intersection_set):
        print(case_number)
else:
    print("Пересечений не найдено.")

---
# 🟡 Очистка и валидация (0-3 балла)
---

### Задание 2.1 — Функция парсинга дат

Напишите функцию `parse_date(date_str)`, которая принимает строку с датой
и возвращает объект `datetime` или `None`, если распарсить не удалось.

Форматы для обработки:
- `"DD.MM.YYYY"` — `strptime`
- `"YYYY-MM-DD"` — `fromisoformat`
- `"DD месяца YYYY г."` — замена русских месяцев + `strptime`
- `"DD/MM/YYYY HH:MM"` — `strptime`

Обрабатывайте `None` и пустые строки через `try/except`.

In [ ]:
from datetime import datetime

def parse_date(date_str):
    """
    Принимает строку с датой и пытается распарсить её в объект datetime.
    Возвращает объект datetime, если успешно, или None, если распарсить не удалось.
    """
    # Обрабатываем None и нестроковые значения
    if not isinstance(date_str, str) or not date_str.strip():
        return None

    # Убираем лишние пробелы по краям
    date_str = date_str.strip()

    try:
        # 1. Формат "DD.MM.YYYY"
        return datetime.strptime(date_str, "%d.%m.%Y")
    except ValueError:
        try:
            # 2. Формат "YYYY-MM-DD"
            return datetime.fromisoformat(date_str)
        except (ValueError, AttributeError):
            try:
                # 3. Формат "DD месяца YYYY г."
                # Создаем словарь для замены русских названий месяцев на их номера
                months_ru = {
                    "января": "01", "февраля": "02", "марта": "03", "апреля": "04",
                    "мая": "05", "июня": "06", "июля": "07", "августа": "08",
                    "сентября": "09", "октября": "10", "ноября": "11", "декабря": "12"
                }
                # Заменяем название месяца на номер
                for ru_month, num_month in months_ru.items():
                    if ru_month in date_str:
                        date_str = date_str.replace(ru_month, num_month)
                        break # Выходим после первой найденной замены
                # Парсим строку без суффикса "г."
                return datetime.strptime(date_str.replace("г.", "").strip(), "%d %m %Y")
            except ValueError:
                try:
                    # 4. Формат "DD/MM/YYYY HH:MM"
                    return datetime.strptime(date_str, "%d/%m/%Y %H:%M")
                except ValueError:
                    # Если ни один из форматов не подошел
                    return None

### Задание 2.2 — Функция валидации сообщения

Напишите функцию `validate_message(msg)`, которая:

1. Проверяет наличие **обязательных полей**: `publisher_inn`, `msg_text`, `date_published`, `type`, `case_number`.
   Если поле отсутствует — ошибка типа `KeyError`.
2. Проверяет, что **ИНН** состоит из 10 или 12 цифр (метод `.isdigit()`).
3. Парсит дату через `parse_date()`. Если дата не парсится — ошибка.
4. Проверяет, что **тип сообщения** — непустая строка.

Функция возвращает кортеж `(valid_msg, errors)`:
- `valid_msg` — словарь с очищенными данными (или `None`)
- `errors` — список строк с описанием ошибок

In [ ]:
def validate_message(msg):
    """
    Валидирует сообщение, проверяя наличие и корректность обязательных полей.

    Возвращает кортеж (valid_msg, errors):
    - valid_msg: словарь с очищенными данными или None, если есть критические ошибки.
    - errors: список строк с описанием найденных ошибок.
    """
    errors = []
    valid_msg = {}

    # 1. Проверка наличия обязательных полей
    required_fields = ['publisher_inn', 'msg_text', 'date_published', 'type', 'case_number']
    for field in required_fields:
        if field not in msg:
            errors.append(f"Ошибка: отсутствует обязательное поле '{field}'")

    # Если критических ошибок нет, продолжаем валидацию
    if not errors:
        # 2. Валидация ИНН (должен быть строкой из 10 или 12 цифр)
        inn = msg['publisher_inn']
        if not (isinstance(inn, str) and inn.isdigit() and len(inn) in (10, 12)):
            errors.append(f"Ошибка: ИНН '{inn}' должен быть строкой из 10 или 12 цифр")
        else:
            valid_msg['publisher_inn'] = inn

        # 3. Валидация текста сообщения (непустая строка)
        msg_text = msg['msg_text']
        if not (isinstance(msg_text, str) and msg_text.strip()):
            errors.append("Ошибка: поле 'msg_text' должно быть непустой строкой")
        else:
            valid_msg['msg_text'] = msg_text

        # 4. Валидация типа сообщения (непустая строка)
        msg_type = msg['type']
        if not (isinstance(msg_type, str) and msg_type.strip()):
            errors.append("Ошибка: поле 'type' должно быть непустой строкой")
        else:
            valid_msg['type'] = msg_type

        # 5. Валидация номера дела (непустая строка)
        case_number = msg['case_number']
        if not (isinstance(case_number, str) and case_number.strip()):
            errors.append("Ошибка: поле 'case_number' должно быть непустой строкой")
        else:
            valid_msg['case_number'] = case_number

        # 6. Парсинг даты
        date_published = msg['date_published']
        parsed_date = parse_date(date_published)
        if parsed_date is None:
            errors.append(f"Ошибка: не удалось распарсить дату '{date_published}'")
        else:
            valid_msg['date_published'] = parsed_date

    # Если были найдены ошибки, возвращаем None и список ошибок
    if errors:
        return None, errors
    else:
        return valid_msg, errors

### Задание 2.3 — Массовая валидация

Примените `validate_message()` ко всем сообщениям.
Разделите результат на два списка: `valid_messages` и `error_records`.

Соберите **статистику ошибок** по типам (сколько `KeyError`, `ValueError` и т.д.).

In [ ]:
from collections import defaultdict

# Инициализация структур для хранения результатов
valid_messages = []
error_records = []
error_stats = defaultdict(int)

# Применение функции валидации к каждому сообщению в исходном списке
for msg in messages:
    valid_msg, errors = validate_message(msg)

    if errors:
        # Запись сообщения с ошибками для отчета
        error_records.append({
            'original_message': msg,
            'errors': errors
        })

        # Классификация и подсчет ошибок
        for err_text in errors:
            if "отсутствует обязательное поле" in err_text:
                error_type = 'KeyError'
            elif "ИНН" in err_text or "не удалось распарсить дату" in err_text or "должен быть непустой строкой" in err_text:
                error_type = 'ValueError'
            else:
                error_type = 'Other'

            error_stats[error_type] += 1
    else:
        # Добавление успешно валидированного сообщения в рабочий список
        valid_messages.append(valid_msg)

# Формирование и вывод итоговой статистики

total_count = len(messages)
valid_count = len(valid_messages)
error_count = len(error_records)

print(f"Всего сообщений: {total_count}")
print(f"Валидных: {valid_count}")
print(f"С ошибками: {error_count}")
print("\nСтатистика ошибок:")
for err_type, count in error_stats.items():
    print(f"{err_type}: {count}")

---
# 🔵 Извлечение данных и аналитика (0-2 балла)
---

### Задание 3.1 — Извлечение сумм из текста

Напишите функцию `extract_amounts(text)`, которая ищет упоминания денежных сумм в тексте сообщения.

Используйте строковые методы.

Ищите по ключевым словам: `"руб."`, `"тыс. руб."`, `"млн руб."`.

Функция возвращает список строк — найденных сумм.

In [ ]:
def extract_amounts(text):
    """
    Ищет упоминания денежных сумм в тексте по ключевым словам:
    "руб.", "тыс. руб.", "млн руб.".
    Возвращает список строк с найденными суммами.
    """
    # Список для хранения найденных сумм
    found_amounts = []

    # Ключевые слова для поиска
    keywords = ["тыс. руб.", "млн руб.", "руб."]

    # Работаем с копией текста в нижнем регистре для регистронезависимого поиска
    text_lower = text.lower()

    for keyword in keywords:
        start_index = 0
        # Ищем все вхождения ключевого слова в тексте
        while True:
            # Находим позицию следующего вхождения ключевого слова
            pos = text_lower.find(keyword, start_index)
            if pos == -1:
                # Ключевое слово больше не найдено, переходим к следующему
                break

            # Начинаем поиск суммы, двигаясь назад от позиции ключевого слова
            amount_start = pos - 1

            # Пропускаем пробелы и запятые перед суммой
            while amount_start >= 0 and text[amount_start] in ' ,':
                amount_start -= 1

            # Собираем символы суммы, пока не встретим букву или начало строки
            amount_chars = []
            while amount_start >= 0 and (text[amount_start].isdigit() or text[amount_start] == ','):
                amount_chars.append(text[amount_start])
                amount_start -= 1

            # Если мы нашли цифры, формируем строку суммы
            if amount_chars:
                # Разворачиваем список символов, убираем запятые и лишние пробелы
                amount_str = ''.join(reversed(amount_chars)).replace(',', '').strip()
                if amount_str: # Проверяем, что строка не пустая
                    found_amounts.append(amount_str)

            # Продолжаем поиск с позиции после текущего найденного ключевого слова
            start_index = pos + len(keyword)

    return found_amounts

### Задание 3.2 — Поиск упоминаний арбитражных судов

Напишите функцию `find_court_mentions(text)`, которая проверяет,
содержит ли текст упоминание арбитражного суда.

Ищите подстроку `"АС "` (с пробелом) в тексте через оператор `in`.
Верните `True` / `False`.

In [ ]:
def find_court_mentions(text):
    """
    Проверяет, содержит ли текст упоминание арбитражного суда.
    Ищет подстроку "АС " (с пробелом).
    Возвращает True, если найдено, иначе False.
    """
    # Проверяем, является ли вход текстовой строкой
    if not isinstance(text, str):
        return False

    # Используем оператор 'in' для поиска подстроки "АС "
    return "АС " in text

### Задание 3.3 — Извлечение ФИО арбитражных управляющих

Напишите функцию `extract_manager_name(text)`, которая ищет ФИО арбитражного управляющего.

Алгоритм:
1. Проверьте, содержит ли текст подстроку `"арбитражный управляющий"`.
2. Если да — найдите позицию этой подстроки, возьмите текст после неё.
3. С помощью `.split()` извлеките следующие 3 слова (ФИО).
4. Соедините через пробел и верните.
5. Если не найдено — верните `None`.

In [ ]:
def extract_manager_name(text):
    """
    Извлекает ФИО арбитражного управляющего из текста.
    Алгоритм: ищет подстроку "арбитражный управляющий", берет текст после нее
    и извлекает следующие 3 слова.
    Возвращает строку с ФИО или None.
    """
    # Проверяем, что на входе строка
    if not isinstance(text, str):
        return None

    # Ключевая фраза для поиска
    key_phrase = "арбитражный управляющий"

    # Проверяем наличие ключевой фразы в тексте (без учета регистра)
    text_lower = text.lower()
    if key_phrase in text_lower:
        # Находим позицию первого вхождения ключевой фразы
        start_pos = text_lower.find(key_phrase)

        # Извлекаем текст, который идет после ключевой фразы
        # Добавляем длину ключевой фразы к стартовой позиции, чтобы начать с конца фразы
        text_after = text[start_pos + len(key_phrase):]

        # Разбиваем оставшийся текст на слова, убирая лишние пробелы
        words = text_after.strip().split()

        # Если в оставшемся тексте есть как минимум 3 слова, собираем их как ФИО
        if len(words) >= 3:
            # Берем первые три слова и соединяем их обратно в строку
            full_name = " ".join(words[:3])
            return full_name

    # Если фраза не найдена или после нее нет 3 слов, возвращаем None
    return None

### Задание 3.4 — Обогащение данных

Для каждого **валидного** сообщения добавьте поля:
- `amounts` — список извлечённых сумм (функция `extract_amounts`)
- `has_court_mention` — `True`/`False` (функция `find_court_mentions`)
- `manager_name` — ФИО или `None` (функция `extract_manager_name`)

In [ ]:
# Проходим по списку валидных сообщений и обогащаем их данными
for msg in valid_messages:
    # Извлекаем суммы из текста сообщения
    msg['amounts'] = extract_amounts(msg['msg_text'])

    # Проверяем наличие упоминания арбитражного суда
    msg['has_court_mention'] = find_court_mentions(msg['msg_text'])

    # Извлекаем ФИО арбитражного управляющего
    msg['manager_name'] = extract_manager_name(msg['msg_text'])

# Выводим результат для проверки (информация по первому сообщению)
if valid_messages:
    print("Обогащение данных завершено.")
    print("\nПример обогащенных данных (первое сообщение):")
    sample = valid_messages[0]
    print(f"Номер дела: {sample.get('case_number')}")
    print(f"Суммы: {sample.get('amounts')}")
    print(f"Упоминание суда: {sample.get('has_court_mention')}")
    print(f"ФИО управляющего: {sample.get('manager_name')}")
else:
    print("Список валидных сообщений пуст. Обогащать нечего.")

### Задание 3.5 — Аналитика

Постройте следующие показатели по **валидным** сообщениям:

1. **Количество сообщений по типам** — словарь `{тип: количество}`
2. **Количество сообщений по регионам** — словарь `{регион: количество}`, пропуская `None`
3. **Топ-5 публикаторов** по количеству сообщений — `sorted(key=lambda...)`
4. **Таймлайн** — сообщения, отсортированные по дате публикации

In [ ]:
from collections import Counter

# 1. Количество сообщений по типам
msg_types = [msg['type'] for msg in valid_messages]
type_counts = dict(Counter(msg_types))

# 2. Количество сообщений по регионам (пропуская None)
regions = [msg['region'] for msg in valid_messages if msg['region'] is not None]
region_counts = dict(Counter(regions))

# 3. Топ-5 публикаторов по количеству сообщений
publishers = [msg['publisher_inn'] for msg in valid_messages]
top_publishers = Counter(publishers).most_common(5)

# 4. Таймлайн — сообщения, отсортированные по дате публикации
# Создаем копию, чтобы не менять исходный список
timeline = sorted(valid_messages, key=lambda x: x['date_published'])

# --- Вывод результатов аналитики ---

print("1. Количество сообщений по типам:")
print(type_counts)
print("\n2. Количество сообщений по регионам (без None):")
print(region_counts)
print("\n3. Топ-5 публикаторов (ИНН: количество сообщений):")
for inn, count in top_publishers:
    print(f"{inn}: {count}")

print("\n4. Таймлайн (первые 3 сообщения по возрастанию даты):")
for msg in timeline[:3]:
    print(f"  Дата: {msg['date_published']}, Номер дела: {msg['case_number']}, Тип: {msg['type']}")

---
# 🟣 Отчётность (0-1 балл)
---

### Задание 4.1 — Подготовка данных для сохранения

Подготовьте данные для записи в JSON. Даты нужно преобразовать обратно в строки (`strftime`),
чтобы JSON был читаемым.

In [ ]:
import copy
from datetime import datetime

# Создаем новый список для данных, готовых к сохранению
prepared_data = []

# Проходим по каждому валидному и обогащенному сообщению
for msg in valid_messages:
    # Создаем поверхностную копию словаря, чтобы не изменять оригинал
    msg_for_save = copy.copy(msg)

    # 1. Преобразуем дату из объекта datetime в строку формата "ГГГГ-ММ-ДД"
    # Это стандартный формат ISO, который легко читается и сортируется
    if 'date_published' in msg_for_save and isinstance(msg_for_save['date_published'], datetime):
        msg_for_save['date_published'] = msg_for_save['date_published'].strftime("%Y-%m-%d")

    # 2. Добавляем подготовленное сообщение в новый список
    prepared_data.append(msg_for_save)

# Выводим результат для проверки (показываем первое подготовленное сообщение)
print("Данные подготовлены к сохранению.")
print("\nПример подготовленного сообщения (первое в списке):")
if prepared_data:
    sample = prepared_data[0]
    print(f"Дата публикации (тип {type(sample['date_published']).__name__}): {sample['date_published']}")
    print(f"Номер дела: {sample.get('case_number')}")
    print(f"Регион: {sample.get('region')}")
    print(f"Суммы: {sample.get('amounts')}")

### Задание 4.2 — Сохранение `analysis_results.json`

Сохраните файл `analysis_results.json` со следующей структурой:
```json
{
  "valid_messages": [...],
  "type_stats": {...},
  "region_stats": {...},
  "priority_messages": [...]
}
```

In [ ]:
import json
import os

# 1. Подготовка словаря с результатами (используем данные из предыдущих шагов)
results_dict = {
    "valid_messages": prepared_data, # Обогащенные и подготовленные сообщения
    "type_stats": type_counts,       # Статистика по типам сообщений
    "region_stats": region_counts,   # Статистика по регионам
    "priority_messages": []          # Список для сообщений, связанных с приоритетными делами
}

# 2. Поиск сообщений, связанных с приоритетными делами
# Используем множество intersection_set, созданное в Задании 1.4
for msg in prepared_data:
    if msg.get('case_number') in intersection_set:
        results_dict['priority_messages'].append(msg)

# 3. Сохранение в файл JSON
output_file = 'analysis_results.json'
try:
    with open(output_file, 'w', encoding='utf-8') as f:
        # Параметр ensure_ascii=False нужен для сохранения кириллицы
        # indent=4 делает JSON-файл удобочитаемым (с отступами)
        json.dump(results_dict, f, ensure_ascii=False, indent=4)
    print(f"Файл успешно сохранен: {os.path.abspath(output_file)}")
except Exception as e:
    print(f"Произошла ошибка при сохранении файла: {e}")

### Задание 4.3 — Сохранение `validation_errors.json`

Сохраните проблемные записи с описанием ошибок.

In [ ]:
import json
import os

# Имя выходного файла
output_file = 'validation_errors.json'

# Проверяем, есть ли что сохранять
if not error_records:
    print("Ошибок валидации не обнаружено. Файл не будет создан.")
else:
    try:
        # Сохраняем список словарей с ошибками в файл
        # ensure_ascii=False для корректного отображения кириллицы
        # indent=4 для читаемого форматирования
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(error_records, f, ensure_ascii=False, indent=4)

        print(f"Файл с ошибками успешно сохранен: {os.path.abspath(output_file)}")
        print(f"Количество сохраненных записей с ошибками: {len(error_records)}")

    except Exception as e:
        # Выводим ошибку, если что-то пошло не так (например, нет прав на запись)
        print(f"Произошла ошибка при сохранении файла: {e}")

### Задание 4.4 — Текстовый отчёт `summary_report.txt`

Сформируйте текстовый аналитический отчёт для руководства с помощью **f-строк**.

Отчёт должен содержать:
- Заголовок и дату
- Общую статистику (всего сообщений, валидных, ошибочных)
- Статистику по типам сообщений
- Статистику по регионам
- Топ-5 публикаторов
- Список приоритетных дел
- Список найденных арбитражных управляющих

In [ ]:
from datetime import datetime

# Собираем все данные для отчёта в одном месте
report_lines = []

# 1. Заголовок и дата
current_date = datetime.now().strftime("%d.%m.%Y %H:%M")
report_lines.append("Аналитический отчёт по данным ЕФРСБ")
report_lines.append(f"Дата формирования: {current_date}")
report_lines.append("=" * 50)

# 2. Общая статистика
report_lines.append("\nОБЩАЯ СТАТИСТИКА")
report_lines.append(f"Всего исходных сообщений: {len(messages)}")
report_lines.append(f"Валидных сообщений: {len(valid_messages)}")
report_lines.append(f"Сообщений с ошибками: {len(error_records)}")

# 3. Статистика по типам сообщений
report_lines.append("\nСТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ")
if type_stats:
    for msg_type, count in type_stats.items():
        report_lines.append(f"  - {msg_type}: {count}")
else:
    report_lines.append("  Данных нет.")

# 4. Статистика по регионам
report_lines.append("\nСТАТИСТИКА ПО РЕГИОНАМ")
if region_stats:
    # Сортируем по убыванию количества для наглядности
    for region, count in sorted(region_stats.items(), key=lambda x: x[1], reverse=True):
        report_lines.append(f"  - {region}: {count}")
else:
    report_lines.append("  Данных нет.")

# 5. Топ-5 публикаторов
report_lines.append("\nТОП-5 ПУБЛИКАТОРОВ ПО КОЛИЧЕСТВУ СООБЩЕНИЙ")
if top_publishers:
    for inn, count in top_publishers:
        report_lines.append(f"  - ИНН {inn}: {count} сообщений")
else:
    report_lines.append("  Данных нет.")

# 6. Список приоритетных дел
report_lines.append("\nПРИОРИТЕТНЫЕ ДЕЛА (пересечение с priority_cases.txt)")
if priority_messages:
    # Собираем уникальные номера дел и сортируем их
    priority_case_numbers = sorted({msg.get('case_number') for msg in priority_messages})
    for case_number in priority_case_numbers:
        report_lines.append(f"  - {case_number}")
else:
    report_lines.append("  Приоритетных дел в данных не обнаружено.")

# 7. Список найденных арбитражных управляющих
# Используем set для сбора уникальных ФИО
manager_names = {msg.get('manager_name') for msg in valid_messages if msg.get('manager_name')}
sorted_managers = sorted(manager_names)

report_lines.append("\nНАЙДЕННЫЕ АРБИТРАЖНЫЕ УПРАВЛЯЮЩИЕ")
if sorted_managers:
    for name in sorted_managers:
        report_lines.append(f"  - {name}")
else:
    report_lines.append("  ФИО арбитражных управляющих в данных не обнаружены.")

# Сохранение отчёта в файл
output_file = 'summary_report.txt'
try:
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(report_lines))
    print(f"Текстовый отчёт успешно сформирован: {output_file}")
except Exception as e:
    print(f"Ошибка при сохранении отчёта: {e}")

---
## ✅ Итоги

Если вы корректно выполнили все 4 уровня, у вас должны получиться:

| Файл | Описание |
|------|----------|
| `analysis_results.json` | Валидные сообщения + статистика + приоритетные дела |
| `validation_errors.json` | Проблемные записи с описанием ошибок |
| `summary_report.txt` | Текстовый аналитический отчёт для руководства |

